In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
spark = SparkSession.builder.appName("Student Data Analysis").getOrCreate()

In [ ]:
filepath='D:\BDA_0024\ABD_LAB\datasets\students.csv'

In [ ]:
df = spark.read.csv(filepath,header=True,inferSchema=True)

In [ ]:
df.show()

+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|       lunch|test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|female|       group B|          bachelor's degree|    standard|                   none|        72|           72|           74|
|female|       group C|               some college|    standard|              completed|        69|           90|           88|
|female|       group B|            master's degree|    standard|                   none|        90|           95|           93|
|  male|       group A|         associate's degree|free/reduced|                   none|        47|           57|           44|
|  male|       group C|               some college|    standard|                   none|        76|     

In [ ]:
df.printSchema()

root
 |-- gender: string (nullable = true)
 |-- race/ethnicity: string (nullable = true)
 |-- parental level of education: string (nullable = true)
 |-- lunch: string (nullable = true)
 |-- test preparation course: string (nullable = true)
 |-- math score: integer (nullable = true)
 |-- reading score: integer (nullable = true)
 |-- writing score: integer (nullable = true)



In [ ]:
# 1. Number of male and female students

gender_count = df \
    .groupBy('gender') \
    .count()

gender_count.show()

+------+-----+
|gender|count|
+------+-----+
|female|  518|
|  male|  482|
+------+-----+



In [ ]:
# 2. List different 'race/ethnicity'
df.select('race/ethnicity') \
  .distinct() \
  .show()

+--------------+
|race/ethnicity|
+--------------+
|       group B|
|       group C|
|       group D|
|       group A|
|       group E|
+--------------+



In [ ]:
# 3. What are different 'parental level of education'?
df.select('parental level of education') \
  .distinct() \
  .show(truncate=False)

+---------------------------+
|parental level of education|
+---------------------------+
|some high school           |
|associate's degree         |
|high school                |
|bachelor's degree          |
|master's degree            |
|some college               |
+---------------------------+



In [ ]:
# 4. How many female students scored more than 79 marks in math, whose parental level of education is 'high school'?
result = df \
    .filter(
        (col('gender') == 'female') &
        (col('math score') > 79) &
        (col('parental level of education') == 'high school')
    )

result.show(truncate=False)

+------+--------------+---------------------------+--------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|lunch   |test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+--------+-----------------------+----------+-------------+-------------+
|female|group B       |high school                |standard|none                   |87        |95           |86           |
|female|group E       |high school                |standard|none                   |99        |93           |90           |
|female|group D       |high school                |standard|completed              |88        |99           |100          |
|female|group B       |high school                |standard|none                   |81        |91           |89           |
|female|group C       |high school                |standard|none                   |81        |84           |82           |
+------+

In [ ]:
count_students = result.count()

print(
    "Number of female students:",
    count_students
)

Number of female students: 5


In [ ]:
# 5. Check whether average maths score of male or female students are high?

avg_math = df \
    .groupBy('gender') \
    .agg(
        round(avg('math score'), 2).alias('Average_Math_Score')
    ) \
    .orderBy(desc('Average_Math_Score'))

avg_math.show()

+------+------------------+
|gender|Average_Math_Score|
+------+------------------+
|  male|             68.73|
|female|             63.63|
+------+------------------+



In [ ]:
# 6. What is average reading score of male and female students?
avg_reading = df \
    .groupBy('gender') \
    .agg(
        round(avg('reading score'), 2).alias('Average_Reading_Score')
    )

avg_reading.show()

+------+---------------------+
|gender|Average_Reading_Score|
+------+---------------------+
|female|                72.61|
|  male|                65.47|
+------+---------------------+



In [ ]:
# 7. Whether students score depends upon 'parental level of education'? Justify your answer.
education_scores = df \
    .groupBy('parental level of education') \
    .agg(
        round(avg('math score'), 2).alias('Average_Math'),
        round(avg('reading score'), 2).alias('Average_Reading'),
        round(avg('writing score'), 2).alias('Average_Writing')
    ) \
    .orderBy(desc('Average_Math'))

education_scores.show(truncate=False)

+---------------------------+------------+---------------+---------------+
|parental level of education|Average_Math|Average_Reading|Average_Writing|
+---------------------------+------------+---------------+---------------+
|master's degree            |69.75       |75.37          |75.68          |
|bachelor's degree          |69.39       |73.0           |73.38          |
|associate's degree         |67.88       |70.93          |69.9           |
|some college               |67.13       |69.46          |68.84          |
|some high school           |63.5        |66.94          |64.89          |
|high school                |62.14       |64.7           |62.45          |
+---------------------------+------------+---------------+---------------+



In [ ]:
# 8. List the records where 'test preparation course' is 'none' and scored more than 70 in maths.
result = df \
    .filter(
        (col('test preparation course') == 'none') &
        (col('math score') > 70)
    )

result.show(truncate=False)

+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|lunch       |test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|female|group B       |bachelor's degree          |standard    |none                   |72        |72           |74           |
|female|group B       |master's degree            |standard    |none                   |90        |95           |93           |
|male  |group C       |some college               |standard    |none                   |76        |78           |75           |
|female|group B       |associate's degree         |standard    |none                   |71        |83           |78           |
|male  |group C       |high school                |standard    |none                   |88        |89   